In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [33]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()
len(docs)

9

In [35]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)


26

In [36]:
embeddings = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-2-preview")

In [37]:
vector_store = Chroma.from_documents(
    documents = splitted_data,
    embedding = embeddings
)

In [13]:
llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash-lite")

In [38]:
def get_context(query:str):
    data = vector_store.similarity_search(query = query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"
        
    return {
        "context": context,
        "question": query
    }

In [28]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answers based on the context for user question.
    if you don't know the answer, then you can say that 'I don't know.'
    Context: {context},
    Question: {question}
""")

In [21]:
rag_chain = get_context | prompt | llm

In [41]:
res = rag_chain.invoke("What is RBC count & is it in range?")

In [42]:
print(res.content)

The RBC count is 4.47 mill/mm3, which is within the normal range of 3.80 - 4.80 mill/mm3.
